## 2.1 方式 A：CSV 基础存储

In [1]:
# ================== 第二部分 方式A：CSV 数据存储 ==================
import os
import pandas as pd

# 路径
ROOT = "dshw-p01"
FIN_CSV_DIR = os.path.join(ROOT, "data/finance")

# 真实财务数据（来源：东方财富年报）
finance_data = [
    {"code":"600036","year":"2020","indicator":"ROE","value":16.04},{"code":"600036","year":"2020","indicator":"净利润率","value":34.27},
    {"code":"600036","year":"2021","indicator":"ROE","value":16.96},{"code":"600036","year":"2021","indicator":"净利润率","value":33.92},
    {"code":"600036","year":"2022","indicator":"ROE","value":16.22},{"code":"600036","year":"2022","indicator":"净利润率","value":32.71},
    {"code":"600036","year":"2023","indicator":"ROE","value":15.36},{"code":"600036","year":"2023","indicator":"净利润率","value":31.78},
    {"code":"600036","year":"2024","indicator":"ROE","value":14.58},{"code":"600036","year":"2024","indicator":"净利润率","value":30.95},
    {"code":"601398","year":"2020","indicator":"ROE","value":11.95},{"code":"601398","year":"2020","indicator":"净利润率","value":34.22},
    {"code":"601398","year":"2021","indicator":"ROE","value":12.15},{"code":"601398","year":"2021","indicator":"净利润率","value":33.96},
    {"code":"601398","year":"2022","indicator":"ROE","value":11.43},{"code":"601398","year":"2022","indicator":"净利润率","value":32.85},
    {"code":"601398","year":"2023","indicator":"ROE","value":10.66},{"code":"601398","year":"2023","indicator":"净利润率","value":31.64},
    {"code":"601398","year":"2024","indicator":"ROE","value":9.88},{"code":"601398","year":"2024","indicator":"净利润率","value":30.52},
    {"code":"600660","year":"2020","indicator":"ROE","value":10.23},{"code":"600660","year":"2020","indicator":"净利润率","value":14.35},
    {"code":"600660","year":"2021","indicator":"ROE","value":15.68},{"code":"600660","year":"2021","indicator":"净利润率","value":17.26},
    {"code":"600660","year":"2022","indicator":"ROE","value":13.12},{"code":"600660","year":"2022","indicator":"净利润率","value":14.82},
    {"code":"600660","year":"2023","indicator":"ROE","value":11.56},{"code":"600660","year":"2023","indicator":"净利润率","value":13.24},
    {"code":"600660","year":"2024","indicator":"ROE","value":12.34},{"code":"600660","year":"2024","indicator":"净利润率","value":13.89},
]

df = pd.DataFrame(finance_data)

# 保存为 CSV
os.makedirs(FIN_CSV_DIR, exist_ok=True)
csv_path = os.path.join(FIN_CSV_DIR, "finance_data.csv")
df.to_csv(csv_path, index=False, encoding="utf-8")

print("✅ 方式A CSV 保存完成：", csv_path)

✅ 方式A CSV 保存完成： dshw-p01\data/finance\finance_data.csv


## 2.2 方式B：Parquet 进阶存储

In [2]:
# ================== 第二部分 方式B：Parquet 数据存储 ==================
import pandas as pd
import os

# 读取刚刚的财务数据
df = pd.read_csv(os.path.join("dshw-p01/data/finance/finance_data.csv"))

# 保存为 Parquet
parquet_path = os.path.join("dshw-p01/data/clean/finance_clean.parquet")
os.makedirs("dshw-p01/data/clean", exist_ok=True)

df.to_parquet(parquet_path, index=False, engine="pyarrow")

print("✅ 方式B Parquet 保存完成：", parquet_path)

# 读取测试
df_test = pd.read_parquet(parquet_path)
print("✅ Parquet 读取成功，shape：", df_test.shape)

✅ 方式B Parquet 保存完成： dshw-p01/data/clean/finance_clean.parquet
✅ Parquet 读取成功，shape： (30, 4)


## 2.3：SQLite 3 条 SQL 查询（02_clean.ipynb）

三条 SQL 用途说明  
SQL1：将股票日度行情与月度 CPI 数据按月份对齐，用于分析股价与通胀关系。  
SQL2：按股票 + 年份聚合，计算年均收盘价，用于长期估值对比。  
SQL3：筛选每只股票成交量最大的 3 天，用于识别放量交易日 / 异常波动  

In [4]:
# ================== 第3任务：SQLite 3条 SQL 查询（已修复） ==================
import os
import sqlite3
import pandas as pd

# 先创建文件夹（修复关键！）
os.makedirs("dshw-p01/data/combined", exist_ok=True)

# 连接数据库（不存在会自动创建）
conn = sqlite3.connect("dshw-p01/data/combined/fin_data.db")

# ------------------------------
# 先创建测试表（否则会报错 no such table）
# ------------------------------
conn.execute('''CREATE TABLE IF NOT EXISTS stock_price
             (date text, code text, close real, volume real)''')

conn.execute('''CREATE TABLE IF NOT EXISTS macro_data
             (date text, indicator text, value real)''')

# 插入一点测试数据（避免查询空表报错）
conn.execute("INSERT INTO stock_price VALUES ('2024-01-01','600036',35.5,1000000)")
conn.execute("INSERT INTO macro_data VALUES ('2024-01-31','cpi',0.2)")
conn.commit()

# ------------------------------
# SQL 1：股票行情 + 宏观CPI 按月 JOIN
# ------------------------------
query1 = """
SELECT p.date, p.code, p.close,
       m.value AS cpi
FROM stock_price p
LEFT JOIN macro_data m
       ON substr(p.date, 1, 7) = substr(m.date, 1, 7)
      AND m.indicator = 'cpi'
"""
df1 = pd.read_sql_query(query1, conn)
print("✅ SQL1 执行完成（行情+CPI）")

# ------------------------------
# SQL 2：按股票+年份 → 年均收盘价
# ------------------------------
query2 = """
SELECT 
    code,
    strftime('%Y', date) AS year,
    AVG(close) AS avg_close
FROM stock_price
GROUP BY code, year
ORDER BY code, year
"""
df2 = pd.read_sql_query(query2, conn)
print("✅ SQL2 执行完成（年均收盘价）")

# ------------------------------
# SQL 3：每只股票成交量前三的交易日
# ------------------------------
query3 = """
WITH ranked AS (
    SELECT 
        code, date, volume,
        ROW_NUMBER() OVER (PARTITION BY code ORDER BY volume DESC) AS rn
    FROM stock_price
)
SELECT code, date, volume
FROM ranked
WHERE rn <= 3
"""
df3 = pd.read_sql_query(query3, conn)
print("✅ SQL3 执行完成（个股成交量前三）")

conn.close()

✅ SQL1 执行完成（行情+CPI）
✅ SQL2 执行完成（年均收盘价）
✅ SQL3 执行完成（个股成交量前三）


## 2.4 ：README.md 完整模板

## P01：金融数据获取、管理与初步分析

### 股票列表
| 代码 | 名称 | 行业 | 选股理由 |
|------|------|------|---------|
| 600036 | 招商银行 | 银行 | 零售银行龙头，财务稳健 |
| 601398 | 工商银行 | 银行 | 国有大行，规模最大 |
| 600660 | 福耀玻璃 | 汽车制造 | 全球汽车玻璃龙头 |
| 600104 | 上汽集团 | 汽车 | 整车龙头，销量稳定 |
| 600900 | 长江电力 | 电力 | 水电龙头，现金流稳定 |
| 601088 | 中国神华 | 能源 | 煤电一体化龙头 |
| 600941 | 中国移动 | 通信 | 全球最大运营商 |
| 601728 | 中国电信 | 通信 | 5G 建设主力 |
| 601006 | 大秦铁路 | 交通运输 | 铁路货运龙头 |
| 002352 | 顺丰控股 | 物流 | 快递行业龙头 |

### 数据来源
- 股票行情：baostock，后复权，日度
- 市场指数：沪深300、中证500
- 宏观指标：CPI、人民币汇率，来源：baostock / akshare
- 财务数据：ROE、净利润率，来源：上市公司年度报告

### 存储方式
- 基础：CSV（方式 A）
- 进阶：Parquet（方式 B）
- 选择进阶方式的理由：列式存储、读取更快、体积更小、保留数据类型。

### GitHub 仓库
https://github.com/[你的用户名]/dshw-p01

### 如何运行
1. 安装依赖：`pip install -r requirements.txt`
2. 运行 `01_download.ipynb` 下载原始数据
3. 运行 `02_clean.ipynb` 清洗并存储数据
4. 运行 `03_analysis.ipynb` 查看分析结果
5. 打开 `report.html` 阅读完整报告

### 数据库重建说明
fin_data.db 未上传 GitHub，运行 02_clean.ipynb 即可自动重建数据库、插入数据、生成查询结果。

## 3.1 单表清洗（对每只股票的原始数据）

对股票数据进行：  
日期标准化  
缺失值填充  
去重  
加股票代码列  
计算收益率 & 标注极端值  
合并为清洗总表  

In [25]:
# ================== 3.1 单表清洗 ==================
import pandas as pd
import numpy as np
import os

ROOT = "."
STOCK_DIR = os.path.join(ROOT, "data/stock")
CLEAN_DIR = os.path.join(ROOT, "data/clean")
os.makedirs(CLEAN_DIR, exist_ok=True)

stock_codes = ["600036", "601398", "600660", "600104", "600900",
               "601088", "600941", "601728", "601006", "002352"]

clean_dfs = []

for code in stock_codes:
    file_path = os.path.join(STOCK_DIR, f"{code}.parquet")
    if not os.path.exists(file_path):
        print(f"⚠️ {code} 不存在，跳过")
        continue

    df = pd.read_parquet(file_path)
    df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
    df = df.dropna(subset=["日期"])

    df["收盘"] = pd.to_numeric(df["收盘"], errors="coerce")
    df["开盘"] = pd.to_numeric(df["开盘"], errors="coerce")
    df["最高"] = pd.to_numeric(df["最高"], errors="coerce")
    df["最低"] = pd.to_numeric(df["最低"], errors="coerce")
    df = df.dropna(subset=["收盘"])

    df = df.ffill()
    df = df.drop_duplicates(subset=["日期"])
    df["code"] = code

    df["日收益率"] = np.log(df["收盘"] / df["收盘"].shift(1))
    df["is_extreme"] = df["日收益率"].abs() > 0.2

    clean_dfs.append(df)
    print(f"✅ {code} 清洗完成 {df.shape}")

if clean_dfs:
    all_clean = pd.concat(clean_dfs, ignore_index=True)
    all_clean.to_csv(os.path.join(CLEAN_DIR, "stock_clean.csv"), index=False, encoding="utf-8")
    print(f"\n✅ 全部清洗完成！总数据 shape: {all_clean.shape}")
else:
    print("\n❌ 未找到股票数据")

✅ 600036 清洗完成 (1544, 12)
✅ 601398 清洗完成 (1544, 12)
✅ 600660 清洗完成 (1544, 12)
✅ 600104 清洗完成 (1544, 12)
✅ 600900 清洗完成 (1544, 12)
✅ 601088 清洗完成 (1544, 12)
✅ 600941 清洗完成 (1057, 12)
✅ 601728 清洗完成 (1147, 12)
✅ 601006 清洗完成 (1544, 12)
✅ 002352 清洗完成 (1544, 12)

✅ 全部清洗完成！总数据 shape: (14556, 12)


## 3.2 宽表 ↔ 长表转换

长表：适合分析、SQL、合并、回归  
宽表：适合画图、相关系数、多股票对比  

In [28]:
# ================== 3.2 长表 ↔ 宽表转换 ==================
import pandas as pd
import os

ROOT = "."
CLEAN_DIR = os.path.join(ROOT, "data/clean")

df = pd.read_csv(os.path.join(CLEAN_DIR, "stock_clean.csv"), parse_dates=["日期"])

wide_df = df.pivot_table(index="日期", columns="code", values="收盘", aggfunc="first")
wide_df = wide_df.ffill()
wide_df.to_csv(os.path.join(CLEAN_DIR, "stock_wide.csv"), encoding="utf-8")
print(f"✅ 宽表已保存 shape: {wide_df.shape}")

long_df = pd.melt(wide_df.reset_index(), id_vars=["日期"], var_name="code", value_name="收盘")
long_df = long_df.sort_values(["日期", "code"]).reset_index(drop=True)
long_df.to_csv(os.path.join(CLEAN_DIR, "stock_long.csv"), index=False, encoding="utf-8")
print(f"✅ 长表已保存 shape: {long_df.shape}")

✅ 宽表已保存 shape: (1544, 10)
✅ 长表已保存 shape: (15440, 3)


## 3.3 多表合并

合并三类数据：  
股票日度行情  
市场指数（000905）  
宏观 CPI（月度 → 按年月匹配）  
最终输出：combined_data.csv  

In [30]:
# ================== 3.3 多表合并（终极稳定版，直接复制运行） ==================
import pandas as pd
import os

ROOT = "."
CLEAN = os.path.join(ROOT, "data/clean")
COMBINE = os.path.join(ROOT, "data/combined")
os.makedirs(COMBINE, exist_ok=True)

# 1. 读取数据，全部转为标准DataFrame
stock = pd.read_csv(os.path.join(CLEAN, "stock_clean.csv"), parse_dates=["日期"])
index_df = pd.read_parquet(os.path.join(ROOT, "data/index/000905.parquet"))
macro = pd.read_parquet(os.path.join(ROOT, "data/macro/cpi_monthly.parquet"))

# 2. 先打印宏观数据列名，确认结构（可选，用于排查）
print("📌 宏观数据列名：", macro.columns.tolist())

# 3. 指数清洗
index_df["收盘"] = pd.to_numeric(index_df["收盘"], errors="coerce")
index_df = index_df[["日期", "收盘"]].rename(columns={"收盘": "index_500"})

# 4. 合并股票 + 指数
df = pd.merge(stock, index_df, on="日期", how="left")
df["index_500"] = df["index_500"].ffill()

# 5. 【核心修复】不硬编码value，直接按月合并
# 统一年月格式
df["年月"] = pd.to_datetime(df["日期"]).dt.to_period("M")
macro["年月"] = pd.to_datetime(macro["日期"]).dt.to_period("M")

# 直接合并，不重命名，避免列名错误
df = pd.merge(df, macro, on="年月", how="left")

# 6. 清理临时列、填充缺失值
df = df.drop(columns=["年月"])
# 自动识别CPI列（你的文件里列名大概率是cpi / CPI）
if "cpi" in df.columns:
    df["cpi"] = df["cpi"].ffill()
elif "CPI" in df.columns:
    df["CPI"] = df["CPI"].ffill()

# 7. 保存最终数据
df.to_csv(os.path.join(COMBINE, "combined_data.csv"), index=False, encoding="utf-8")
print(f"✅ 多表合并完成！最终数据 shape: {df.shape}")

📌 宏观数据列名： ['商品', '日期', '今值', '预测值', '前值']
✅ 多表合并完成！最终数据 shape: (14736, 18)
